# GPU-Parallel Detection of Temporal Hypergraph Motifs

<hr>

In [3]:
#!pip install numba
#!pip install hypernetx

In [4]:
import numpy as np
import networkx as nx
import hypernetx as hnx
import matplotlib.pyplot as plt
from numba import cuda, uint64, int32

In [5]:
@cuda.jit
def motif_kernel(E1, E2, E3, m1, m2, m3,
                 results_i, results_j, results_k, counter):

    idx = cuda.grid(1)
    total_pairs = m1 * m2
    
    if idx >= total_pairs:
        return

    i = idx // m2
    j = idx % m2

    e1 = E1[i]
    e2 = E2[j]

    # Overlap condition e1 ∩ e2 ≠ ∅
    if (e1 & e2) == 0:
        return

    # Subset constraints
    if ((e1 & ~e2) == 0) or ((e2 & ~e1) == 0):
        return

    for k in range(m3):
        e3 = E3[k]

        #Overlap e2 ∩ e3 ≠ ∅
        if (e2 & e3) == 0:
            continue

        #No subset relations
        if ((e1 & ~e3) == 0) or ((e3 & ~e1) == 0):
            continue

        if ((e2 & ~e3) == 0) or ((e3 & ~e2) == 0):
            continue

        #Atomic write
        pos = cuda.atomic.add(counter, 0, 1)

        results_i[pos] = i
        results_j[pos] = j
        results_k[pos] = k


In [6]:
def find_motifs_gpu(E1_host, E2_host, E3_host):
    """Find THMs.
    """
    m1 = len(E1_host)
    m2 = len(E2_host)
    m3 = len(E3_host)

    E1 = cuda.to_device(np.array(E1_host, dtype=np.uint64))
    E2 = cuda.to_device(np.array(E2_host, dtype=np.uint64))
    E3 = cuda.to_device(np.array(E3_host, dtype=np.uint64))

    max_possible = m1 * m2 * m3  # upper bound
    results_i = cuda.device_array(max_possible, dtype=np.int32)
    results_j = cuda.device_array(max_possible, dtype=np.int32)
    results_k = cuda.device_array(max_possible, dtype=np.int32)

    counter = cuda.to_device(np.array([0], dtype=np.int32))

    threads_per_block = 256
    blocks = (m1 * m2 + threads_per_block - 1) // threads_per_block

    motif_kernel[blocks, threads_per_block](
        E1, E2, E3,
        m1, m2, m3,
        results_i, results_j, results_k,
        counter
    )

    total = counter.copy_to_host()[0]

    return (results_i.copy_to_host()[:total],
            results_j.copy_to_host()[:total],
            results_k.copy_to_host()[:total],)


In [7]:
def lists_to_tuples(list_of_lists):
    return [tuple(lst) for lst in list_of_lists]


def to_hnx(edges_t):
    edge_dict = {f"e{i}": list(e) for i, e in enumerate(edges_t)}
    return hnx.Hypergraph(edge_dict)


def incidence_to_hyperedges(incidence):
    """Convert incidence matrix (nodes x hyperedges)
    into a list of hyperedges (tuples of nodes).
    """
    hyperedges = []
    for j in range(incidence.shape[1]):
        nodes = tuple(np.where(incidence[:, j] != 0)[0])
        hyperedges.append(nodes)
    return hyperedges


def hnx_to_incidence_matrix(H):
    """Convert a HyperNetX hypergraph to its incidence matrix.
    H: HyperNetX hypergraph
    Returns: numpy.ndarray (n_nodes x n_edges)
        node_index : dict (node -> row index)
        edge_index : dict (edge -> column index)
    """
    nodes = list(H.nodes)
    edges = list(H.edges)

    node_index = {v: i for i, v in enumerate(nodes)}
    edge_index = {e: j for j, e in enumerate(edges)}

    B = np.zeros((len(nodes), len(edges)), dtype=int)

    for e in edges:
        j = edge_index[e]
        for v in H.edges[e]:
            i = node_index[v]
            B[i, j] = 1

    return B


def cliques_gtoet_k(C, k):
    cliques = []
    for cliq in C:
        if len(cliq) >= k:
            cliques.append(cliq)
    return cliques


def loopless_graph(M):
    G = nx.from_numpy_array(M, create_using=nx.Graph())
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


In [8]:
def diag_zero(M):
    '''
    M: an adjacency matrix.
    Return: M with a null diagonal.
    '''
    for i in range(len(M)):
        M[i, i] = 0
    return M


def directed_erdos_renyi_Gnp_model(n, p, weight=False):
    """Returns a binomial random digraph.
    Parameters:
    ----------
    n : (integer) Number of nodes.
    p : (float) Probability of adding a directed edge (i,j).
    weight: (boolean) If True, it randomly assigns weights to edges.
    """
    A = np.zeros((n,n))  #null matrix (no connections).
    for i in range(n):
        for j in range(n):
            if i != j:
                A[i, j] = np.random.binomial(1, p) #add a directed edge with probability p.

    if weight==True:
        W = random_weights(A)
        return diag_zero(W)
    else:
        return diag_zero(A)


def include_exclude_edges(L, p1, p2):
    '''Returns an adjacency matrix with some different edges.

    Parameters
    ----------
    L: The underlying adjacency matrix.
    p1: probability that edge (i,j) exists at time t.
    p2: probability that a non-existent edge (i,j) arises at time t.
    '''
    M = np.zeros((len(L), len(L)))
    for i in range (len(L)):
        for j in range (len(L)):
            if L[i, j] != 0:
                M[i, j] = np.random.binomial(1, p1)*np.random.uniform(0.00001, 1)
            if L[i, j] == 0:
                M[i, j] = np.random.binomial(1, p2)*np.random.uniform(0.00001, 1)
    return diag_zero(M)


def dynamic_random_digraph_model(M, p1, p2, t):
    '''Returns a weighted dynamic random digraph.

    Parameters
    ----------
    L: The underlying adjacent matrix
    p1: probability that edge (i,j) exists at time t.
    p2: probability that a non-existent edge (i,j) arises at time t.
    t: time range.
    '''
    D=[]
    for i in range(t):
        D.append(np.zeros((len(M),len(M))))

    D[0]=M
    for k in range(1,t):
        D[k] = include_exclude_edges(D[k-1], p1, p2)
    return D

In [22]:
M = directed_erdos_renyi_Gnp_model(900, 0.01, weight=False)

adjacency_matrices = dynamic_random_digraph_model(M, 0.04, 0.04, 4)

In [23]:
G1 = loopless_graph(adjacency_matrices[0])
G2 = loopless_graph(adjacency_matrices[1])
G3 = loopless_graph(adjacency_matrices[2])
G4 = loopless_graph(adjacency_matrices[3])
#G5 = loopless_graph(adjacency_matrices[4])

k = 2
Events=[]
cliques1 = cliques_gtoet_k(list(nx.find_cliques(G1)), k)
cliques2 = cliques_gtoet_k(list(nx.find_cliques(G2)), k)
cliques3 = cliques_gtoet_k(list(nx.find_cliques(G3)), k)
cliques4 = cliques_gtoet_k(list(nx.find_cliques(G4)), k)
#cliques5 = cliques_gtoet_k(list(nx.find_cliques(G5)), k)

Events.append(lists_to_tuples(cliques1))
Events.append(lists_to_tuples(cliques2))
Events.append(lists_to_tuples(cliques3))
Events.append(lists_to_tuples(cliques4))
#Events.append(lists_to_tuples(cliques5))

H_hnx1 = to_hnx(Events[0])
H_hnx2 = to_hnx(Events[1])
H_hnx3 = to_hnx(Events[2])
H_hnx4 = to_hnx(Events[3])

'''
fig, axes = plt.subplots(1, 4, figsize=(11, 4))
hnx.draw(H_hnx1, pos=nx.circular_layout(G1), ax=axes[0], with_node_labels=False, with_edge_labels=True)
hnx.draw(H_hnx2, pos=nx.circular_layout(G2), ax=axes[1], with_node_labels=False, with_edge_labels=True)
hnx.draw(H_hnx3, pos=nx.circular_layout(G3), ax=axes[2], with_node_labels=False, with_edge_labels=True)
hnx.draw(H_hnx4, pos=nx.circular_layout(G4), ax=axes[3], with_node_labels=False, with_edge_labels=True)
#plt.title("Temporal Hypergraph")
plt.show()
'''


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'\nfig, axes = plt.subplots(1, 4, figsize=(11, 4))\nhnx.draw(H_hnx1, pos=nx.circular_layout(G1), ax=axes[0], with_node_labels=False, with_edge_labels=True)\nhnx.draw(H_hnx2, pos=nx.circular_layout(G2), ax=axes[1], with_node_labels=False, with_edge_labels=True)\nhnx.draw(H_hnx3, pos=nx.circular_layout(G3), ax=axes[2], with_node_labels=False, with_edge_labels=True)\nhnx.draw(H_hnx4, pos=nx.circular_layout(G4), ax=axes[3], with_node_labels=False, with_edge_labels=True)\n#plt.title("Temporal Hypergraph")\nplt.show()\n'

In [ ]:
find_motifs_gpu(H_hnx1, H_hnx2, H_hnx3)

In [21]:
print(nx.density(G2))
print(G2.number_of_edges())

0.0786107634543179
25124
